# Legacy Stack tiered evaluator (not for the current Lift v2 model)

This notebook is retained only as a record of the earlier Stack experiment. Its old architecture and one-action predictor contract are intentionally incompatible with the corrected Lift v2 checkpoints. Use `run_primary.ipynb` for current evaluation; do not use this notebook for the new training run.


In [ ]:
%pip install -q h5py imageio imageio-ffmpeg mujoco==3.2.4 robosuite==1.4.1 git+https://github.com/NVlabs/mimicgen.git@72bd767c255545f462e7ccfb2731f2e5d4c1d9bb

In [ ]:
import ctypes
import io
import re
import time
import zipfile
from collections import deque
from pathlib import Path

import h5py
import imageio.v2 as imageio
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

from google.colab import drive
drive.mount('/content/drive')

PRIMARY_DIR = Path('/content/drive/MyDrive/primary_split')
MODEL_DIR = PRIMARY_DIR / 'models'
ROLLOUT_DIR = PRIMARY_DIR / 'rollouts'
ROLLOUT_DIR.mkdir(parents=True, exist_ok=True)

WINDOW = 3
EMBED_DIM = 192
ACTION_DIM = 7
HORIZON = 5
NUM_SAMPLES = 300
NUM_ELITES = 30
CEM_ITERS = 10
GRASP_CONFIRM_STEPS = 2
CONTROL_FREQ = 20
VIDEO_SIZE = 256

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {DEVICE}')

def select_checkpoint(model_dir: Path):
    best_checkpoint = model_dir / 'best_model.zip'
    if best_checkpoint.exists():
        return best_checkpoint, 'best validation model'

    checkpoints = list(model_dir.glob('epoch_*_model.zip'))
    if not checkpoints:
        raise FileNotFoundError(
            f'No best_model.zip or epoch_*_model.zip files found in {model_dir}'
        )

    def epoch_number(path: Path) -> int:
        match = re.search(r'epoch_(\d+)_model\.zip$', path.name)
        return int(match.group(1)) if match else -1

    checkpoint = max(checkpoints, key=epoch_number)
    return checkpoint, f'latest epoch {epoch_number(checkpoint)} (fallback)'

CHECKPOINT, CHECKPOINT_LABEL = select_checkpoint(MODEL_DIR)
print(f'selected checkpoint: {CHECKPOINT} ({CHECKPOINT_LABEL})')

eval_candidates = [
    PRIMARY_DIR / 'evals.hdf5',
    Path('/content/data/robot/mimicgen/evals.hdf5'),
]
EVAL_HDF5 = next((path for path in eval_candidates if path.exists()), None)
if EVAL_HDF5 is None:
    raise FileNotFoundError(
        'Could not find evals.hdf5. Put it in /content/drive/MyDrive/primary_split/ '
        'or update EVAL_HDF5.'
    )
print(f'exact-reset eval HDF5: {EVAL_HDF5}')

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=192, nheads=16, dropout=0.1, causal_masking=True):
        super().__init__()
        self.d_model = d_model
        self.d_k = d_model // nheads
        self.nheads = nheads
        self.causal_masking = causal_masking
        self.dropout = dropout
        self.output_dropout = nn.Dropout(dropout)
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x):
        batch, length, _ = x.shape
        q = self.W_q(x).reshape(batch, length, self.nheads, self.d_k).transpose(1, 2)
        k = self.W_k(x).reshape(batch, length, self.nheads, self.d_k).transpose(1, 2)
        v = self.W_v(x).reshape(batch, length, self.nheads, self.d_k).transpose(1, 2)
        output = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=self.causal_masking,
        )
        output = output.transpose(1, 2).reshape(batch, length, self.d_model)
        return self.output_dropout(self.W_o(output))


class TransformerBlock(nn.Module):
    def __init__(self, d_model=192, nheads=16, d_action=None, dropout=0.1, causal_masking=True, adaptive=True):
        super().__init__()
        self.adaptive = adaptive
        self.norm = nn.LayerNorm(d_model, elementwise_affine=not adaptive)
        if adaptive:
            self.W_LNparams = nn.Linear(d_action, 4 * d_model)
            self.W_LNparams.weight.data.zero_()
            self.W_LNparams.bias.data.zero_()
        self.attn_block = MultiHeadAttention(d_model, nheads, dropout, causal_masking)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, z, actions=None):
        if self.adaptive:
            scale_1, shift_1, scale_2, shift_2 = self.W_LNparams(actions).chunk(4, dim=-1)
            z = z + self.attn_block((1 + scale_1) * self.norm(z) + shift_1)
            z = z + self.mlp((1 + scale_2) * self.norm(z) + shift_2)
        else:
            z = z + self.attn_block(self.norm(z))
            z = z + self.mlp(self.norm(z))
        return z


class CustomVitTiny(nn.Module):
    def __init__(self, patch=14, nheads=3, nlayers=12, d_model=192, npatches=36):
        super().__init__()
        self.d_model = d_model
        self.npatches = npatches
        self.conv_flattener = nn.Conv2d(3, d_model, kernel_size=patch, stride=patch)
        self.transformer_layers = nn.ModuleList([
            TransformerBlock(d_model, nheads, dropout=0.0, causal_masking=False, adaptive=False)
            for _ in range(nlayers)
        ])
        self.position_emb = nn.Parameter(torch.randn(npatches + 1, d_model))
        self.cls_emb = nn.Parameter(torch.randn(1, 1, 1, d_model))

    def forward(self, x):
        batch, window, channels, height, width = x.shape
        x = self.conv_flattener(x.reshape(batch * window, channels, height, width))
        x = x.reshape(batch, window, self.d_model, self.npatches).permute(0, 1, 3, 2)
        cls = self.cls_emb.expand(batch, window, 1, self.d_model)
        x = (torch.cat([cls, x], dim=2) + self.position_emb).reshape(batch * window, self.npatches + 1, self.d_model)
        for layer in self.transformer_layers:
            x = layer(x)
        return x.reshape(batch, window, self.npatches + 1, self.d_model)[:, :, 0, :]


class EncoderNetwork(nn.Module):
    def __init__(self, patch=14, nheads=3, nlayers=12, d_model=192, npatches=36):
        super().__init__()
        self.d_model = d_model
        self.vit = CustomVitTiny(patch, nheads, nlayers, d_model, npatches)
        self.projector = nn.Linear(d_model, d_model)
        self.batch_norm = nn.BatchNorm1d(d_model)

    def forward(self, x):
        batch, window, _, _, _ = x.shape
        z = self.vit(x).reshape(batch * window, self.d_model)
        z = self.batch_norm(self.projector(z))
        return z.reshape(batch, window, self.d_model)


class RobotEncoder(nn.Module):
    def __init__(self, patch=14, nheads=3, nlayers=12, d_model=192, d_state=9, d_state_emb=64, npatches=36):
        super().__init__()
        self.encoder = EncoderNetwork(patch, nheads, nlayers, d_model, npatches)
        self.state_encoder = nn.Sequential(nn.Linear(d_state, 2 * d_state), nn.GELU(), nn.Linear(2 * d_state, d_state_emb))
        self.fusion_mlp = nn.Sequential(
            nn.Linear(2 * d_model + d_state_emb, 2 * d_model),
            nn.GELU(),
            nn.Linear(2 * d_model, d_model),
        )

    def forward(self, above, wrist, state):
        batch_size = above.shape[0]
        view_emb = self.encoder(torch.cat((above, wrist), dim=0))
        above_emb, wrist_emb = view_emb.split(batch_size, dim=0)
        state_emb = self.state_encoder(state)
        return self.fusion_mlp(torch.cat([above_emb, wrist_emb, state_emb], dim=-1))


class PredictorNetwork(nn.Module):
    def __init__(self, d_model=192, nheads=16, d_action=7, dropout=0.1, nlayers=6, window=3):
        super().__init__()
        self.d_model = d_model
        self.transformer_layers = nn.ModuleList([
            TransformerBlock(d_model, nheads, d_action, dropout, True, True)
            for _ in range(nlayers)
        ])
        self.position_emb = nn.Parameter(torch.randn(window, d_model))
        self.projector = nn.Linear(d_model, d_model)
        self.batch_norm = nn.BatchNorm1d(d_model)

    def forward(self, z, actions):
        batch, window, _ = z.shape
        z = z + self.position_emb
        for layer in self.transformer_layers:
            z = layer(z, actions)
        z = self.batch_norm(self.projector(z.reshape(batch * window, self.d_model)))
        return z.reshape(batch, window, self.d_model)

In [ ]:
class Planner:
	def __init__(
		self,
		device,
		enc: nn.Module,
		pred: nn.Module,
		H: int = 5,
		N: int = 300,
		K: int = 30,
		T: int = 10,
		action_dim: int = 7,
		action_limit: float | None = None,
		discrete_gripper: bool = False,
		action_mean=None,
		action_std=None,
		action_low=None,
		action_high=None,
		min_std=0.05,
		momentum: float = 0.1,
		seed: int | None = None,
	):
		self.device = torch.device(device)
		self.enc = enc.to(self.device)
		self.pred = pred.to(self.device)
		self.H = H
		self.N = N
		self.K = K
		self.T = T
		self.action_dim = action_dim
		self.action_limit = action_limit
		self.discrete_gripper = discrete_gripper

		if H < 1 or N < 1 or K < 1 or T < 1:
			raise ValueError("H, N, K, and T must all be positive")
		if K > N:
			raise ValueError(f"K ({K}) cannot exceed N ({N})")

		self.action_mean = self._action_vector(action_mean, 0.0, "action_mean")
		self.action_std = self._action_vector(action_std, 1.0, "action_std")
		self.min_std = self._action_vector(min_std, 0.05, "min_std")
		if np.any(self.action_std <= 0) or np.any(self.min_std <= 0):
			raise ValueError("action_std and min_std must be positive")
		if not 0.0 <= momentum < 1.0:
			raise ValueError(f"momentum must be in [0, 1), got {momentum}")
		self.momentum = float(momentum)

		if action_low is None and action_limit is not None:
			action_low = -float(action_limit)
		if action_high is None and action_limit is not None:
			action_high = float(action_limit)
		self.action_low = (
			None
			if action_low is None
			else self._action_vector(action_low, -1.0, "action_low")
		)
		self.action_high = (
			None
			if action_high is None
			else self._action_vector(action_high, 1.0, "action_high")
		)
		if (self.action_low is None) != (self.action_high is None):
			raise ValueError("action_low and action_high must be provided together")
		if self.action_low is not None:
			if np.any(self.action_low >= self.action_high):
				raise ValueError("Every action_low must be below action_high")
			self.action_mean = np.clip(
				self.action_mean,
				self.action_low,
				self.action_high,
			)
		if self.discrete_gripper and self.action_low is None:
			raise ValueError(
				"discrete_gripper requires action bounds or action_limit"
			)

		self.sampler = np.random.default_rng(seed)

	def _action_vector(self, value, default, name):
		if value is None:
			value = default
		array = np.asarray(value, dtype=np.float32)
		if array.ndim == 0:
			array = np.full(self.action_dim, float(array), dtype=np.float32)
		if array.shape != (self.action_dim,):
			raise ValueError(
				f"{name} must be scalar or shape ({self.action_dim},), got {array.shape}"
			)
		if not np.isfinite(array).all():
			raise ValueError(f"{name} contains non-finite values")
		return array.copy()

	def shifted_plan(self, plan):
		"""Shift a plan by one action for receding-horizon warm start."""
		plan = np.asarray(plan, dtype=np.float32)
		if plan.shape != (self.H, self.action_dim):
			raise ValueError(
				f"plan must have shape {(self.H, self.action_dim)}, got {plan.shape}"
			)
		return np.concatenate((plan[1:], self.action_mean[None]), axis=0)

	def _prepare_fixed_gripper(self, fixed_gripper, device):
		if fixed_gripper is None:
			return None
		gripper = torch.as_tensor(fixed_gripper, dtype=torch.float32, device=device)
		if gripper.ndim == 0:
			gripper = gripper.expand(self.H)
		if gripper.shape != (self.H,):
			raise ValueError(f"fixed_gripper must be scalar or shape ({self.H},), got {tuple(gripper.shape)}")
		if not torch.isfinite(gripper).all():
			raise ValueError("fixed_gripper contains non-finite values")
		return torch.where(gripper >= 0, float(self.action_high[-1]), float(self.action_low[-1]))

	def _bound_actions(self, actions, fixed_gripper=None):
		if self.action_low is not None:
			low = torch.as_tensor(self.action_low, device=actions.device)
			high = torch.as_tensor(self.action_high, device=actions.device)
			actions = torch.maximum(torch.minimum(actions, high), low)
		if self.discrete_gripper:
			actions[..., -1] = torch.where(
				actions[..., -1] >= 0,
				float(self.action_high[-1]),
				float(self.action_low[-1]),
			)
		if fixed_gripper is not None:
			if not self.discrete_gripper:
				raise ValueError("fixed_gripper requires discrete_gripper=True")
			actions[..., -1] = self._prepare_fixed_gripper(fixed_gripper, actions.device)
		return actions

	def _prepare_inputs(self, zi, a, zg):
		zi = zi.to(self.device, dtype=torch.float32)
		a = a.to(self.device, dtype=torch.float32)
		zg = zg.to(self.device, dtype=torch.float32)
		if zg.ndim == 1:
			zg = zg.unsqueeze(0)
		if zg.ndim != 2:
			raise ValueError(
				f"zg must have shape (D,) or (G, D), got {tuple(zg.shape)}"
			)
		return zi, a, zg

	@torch.no_grad()
	def score_actions(self, zi, a, actions, zg, apply_bounds=True):
		"""Return terminal latent goal cost for each supplied action sequence."""
		zi, a, zg = self._prepare_inputs(zi, a, zg)
		actions = torch.as_tensor(
			actions,
			dtype=torch.float32,
			device=self.device,
		)
		if actions.ndim == 2:
			actions = actions.unsqueeze(0)
		if actions.ndim != 3 or actions.shape[2] != self.action_dim:
			raise ValueError(
				"actions must have shape (H, A) or (N, H, A), got "
				f"{tuple(actions.shape)}"
			)
		if apply_bounds:
			actions = self._bound_actions(actions)

		num_sequences = actions.shape[0]
		window = zi.unsqueeze(0).expand(num_sequences, -1, -1)
		dummy = torch.zeros((1, a.shape[1]), dtype=a.dtype, device=a.device)
		action_window = torch.cat((dummy, a), dim=0).unsqueeze(0)
		action_window = action_window.expand(num_sequences, -1, -1)

		for step in range(actions.shape[1]):
			action_window = torch.cat(
				(action_window[:, 1:], actions[:, step:step + 1]),
				dim=1,
			)
			prediction = self.pred(window, action_window)[:, -1]
			window = torch.cat((window[:, 1:], prediction.unsqueeze(1)), dim=1)

		goal_costs = (
			window[:, -1].unsqueeze(1) - zg.unsqueeze(0)
		).square().mean(dim=-1)
		return goal_costs.min(dim=1).values

	@torch.no_grad()
	def CEM(self, zi, a, zg, initial_mean=None, fixed_gripper=None):
		"""Optimize and return a complete ``H x action_dim`` action plan."""
		zi, a, zg = self._prepare_inputs(zi, a, zg)
		self.pred.eval()
		self.enc.eval()

		if initial_mean is None:
			mean = np.broadcast_to(
				self.action_mean,
				(self.H, self.action_dim),
			).copy()
		else:
			mean = np.asarray(initial_mean, dtype=np.float32).copy()
			if mean.shape != (self.H, self.action_dim):
				raise ValueError(
					f"initial_mean must have shape {(self.H, self.action_dim)}, "
					f"got {mean.shape}"
				)
			if self.action_low is not None:
				mean = np.clip(mean, self.action_low, self.action_high)
		std = np.broadcast_to(
			self.action_std,
			(self.H, self.action_dim),
		).copy()
		if fixed_gripper is not None:
			gripper = self._prepare_fixed_gripper(fixed_gripper, self.device).cpu().numpy()
			mean[:, -1] = gripper

		for _ in range(self.T):
			actions = self.sampler.normal(
				mean,
				std,
				size=(self.N, self.H, self.action_dim),
			)
			actions = torch.as_tensor(
				actions,
				dtype=torch.float32,
				device=self.device,
			)
			actions = self._bound_actions(actions, fixed_gripper=fixed_gripper)
			costs = self.score_actions(zi, a, actions, zg, apply_bounds=False)
			elite_indices = torch.topk(costs, k=self.K, largest=False).indices
			elites = actions[elite_indices]

			elite_mean = elites.mean(dim=0).cpu().numpy()
			elite_std = elites.std(dim=0, unbiased=False).cpu().numpy()
			mean = self.momentum * mean + (1.0 - self.momentum) * elite_mean
			std = self.momentum * std + (1.0 - self.momentum) * elite_std
			std = np.maximum(std, self.min_std)

		if self.action_low is not None:
			mean = np.clip(mean, self.action_low, self.action_high)
		if self.discrete_gripper:
			mean[..., -1] = np.where(
				mean[..., -1] >= 0,
				self.action_high[-1],
				self.action_low[-1],
			)
		if fixed_gripper is not None:
			mean[:, -1] = gripper
		return mean.astype(np.float32, copy=False)

def load_models(checkpoint: Path, device):
    encoder = RobotEncoder().to(device)
    predictor = PredictorNetwork(d_action=ACTION_DIM, window=WINDOW).to(device)
    with zipfile.ZipFile(checkpoint, 'r') as archive:
        with archive.open('encoder_weights.pth') as weights:
            encoder.load_state_dict(torch.load(io.BytesIO(weights.read()), map_location=device, weights_only=True))
        with archive.open('predictor_weights.pth') as weights:
            predictor.load_state_dict(torch.load(io.BytesIO(weights.read()), map_location=device, weights_only=True))
        if 'state_stats.pth' not in archive.namelist():
            raise ValueError('Checkpoint uses the old action/state contract; retrain before inference')
        with archive.open('state_stats.pth') as stats_file:
            state_stats = torch.load(io.BytesIO(stats_file.read()), map_location='cpu', weights_only=True)
    state_mean = state_stats['state_mean'].float()
    state_std = state_stats['state_std'].float()
    if state_mean.shape != (9,) or state_std.shape != (9,):
        raise ValueError('Checkpoint state statistics must both have shape (9,)')
    if not torch.isfinite(state_mean).all() or not torch.isfinite(state_std).all() or (state_std <= 0).any():
        raise ValueError('Checkpoint state statistics contain invalid values')
    encoder.eval()
    predictor.eval()
    return encoder, predictor, state_mean, state_std

encoder, predictor, STATE_MEAN, STATE_STD = load_models(CHECKPOINT, DEVICE)


In [ ]:
CAM_ABOVE = 'agentview'
CAM_WRIST = 'robot0_eye_in_hand'
IMG_SIZE = 84

import mujoco
import robosuite as suite
import robosuite.macros as macros
from robosuite.controllers import load_controller_config
import mimicgen  # registers Stack_D0 with robosuite

# Training HDF5 images use the OpenCV (upright) convention.
macros.IMAGE_CONVENTION = 'opencv'

def set_geom_groups(env, collision=False, visual=True):
    context = env.sim._render_context_offscreen
    if context is None:
        return
    address = context.vopt.geomgroup.__array_interface__['data'][0]
    groups = (ctypes.c_uint8 * 6).from_address(address)
    for index in range(6):
        groups[index] = 0
    groups[0] = int(collision)
    groups[1] = int(visual)

def build_robot_state(pos, quat, grip):
    quat = quat / quat.norm(dim=-1, keepdim=True).clamp_min(1e-8)
    # Preserve the temporally continuous quaternion sign.
    return torch.cat((pos, quat, grip), dim=-1)

def normalize_robot_state(state):
    return (state - STATE_MEAN) / STATE_STD

def obs_to_tensors(obs, device):
    above = torch.from_numpy(obs[f'{CAM_ABOVE}_image']).float().permute(2, 0, 1) / 255.0
    wrist = torch.from_numpy(obs[f'{CAM_WRIST}_image']).float().permute(2, 0, 1) / 255.0
    state = build_robot_state(
        torch.from_numpy(obs['robot0_eef_pos']).float(),
        torch.from_numpy(obs['robot0_eef_quat']).float(),
        torch.from_numpy(obs['robot0_gripper_qpos']).float(),
    )
    state = normalize_robot_state(state)
    return (above[None, None].to(device), wrist[None, None].to(device), state[None, None].to(device))

@torch.no_grad()
def encode_obs(encoder, obs, device):
    above, wrist, state = obs_to_tensors(obs, device)
    return encoder(above, wrist, state).squeeze(0).squeeze(0)

@torch.no_grad()
def encode_demo_frames(obs, frame_indices, device, batch_size=128):
    embeddings = []
    for offset in range(0, len(frame_indices), batch_size):
        indices = frame_indices[offset:offset + batch_size]
        above = torch.from_numpy(obs['agentview_image'][indices]).float().permute(0, 3, 1, 2).unsqueeze(1) / 255.0
        wrist = torch.from_numpy(obs['robot0_eye_in_hand_image'][indices]).float().permute(0, 3, 1, 2).unsqueeze(1) / 255.0
        state = build_robot_state(
            torch.from_numpy(obs['robot0_eef_pos'][indices]).float(),
            torch.from_numpy(obs['robot0_eef_quat'][indices]).float(),
            torch.from_numpy(obs['robot0_gripper_qpos'][indices]).float(),
        )
        state = normalize_robot_state(state)
        embeddings.append(encoder(above.to(device), wrist.to(device), state.unsqueeze(1).to(device))[:, 0].cpu())
    return torch.cat(embeddings).to(device)

@torch.no_grad()
def load_tiered_eval_bank(encoder, path, device, demo_name=None):
    starts, grasps, goals = [], [], []
    names, initial_states, model_xmls = [], [], []
    with h5py.File(path, 'r') as file:
        collected_mujoco = file['data'].attrs.get('mujoco_version')
        if collected_mujoco is not None:
            collected_mujoco = collected_mujoco.decode('utf-8') if isinstance(collected_mujoco, bytes) else str(collected_mujoco)
            if collected_mujoco != mujoco.__version__:
                print(f'WARNING: evals use MuJoCo {collected_mujoco}; inference uses {mujoco.__version__}')
        if demo_name is not None:
            demo_names = [demo_name]
        elif 'mask' in file and 'eval' in file['mask']:
            demo_names = [raw.decode('utf-8') if isinstance(raw, bytes) else str(raw) for raw in file['mask/eval'][:]]
        else:
            demo_names = sorted(file['data'].keys())
        if not demo_names:
            raise ValueError(f'No eval demos found in {path}')

        for name in demo_names:
            demo = file[f'data/{name}']
            initial_index = int(demo.attrs.get('initial_index', 0))
            grasp_index = int(demo.attrs.get('grasp_index', -1))
            goal_index = int(demo.attrs.get('goal_index', -1))
            if initial_index != 0 or not (0 < grasp_index < goal_index < demo['states'].shape[0]):
                raise ValueError(f'{name} has invalid exact-eval frame indices')
            frame_indices = [initial_index, grasp_index, goal_index]
            embeddings = encode_demo_frames(demo['obs'], frame_indices, device)
            starts.append(embeddings[0].cpu())
            grasps.append(embeddings[1].cpu())
            goals.append(embeddings[2].cpu())
            names.append(name)
            initial_states.append(np.asarray(demo['states'][initial_index]).copy())
            model_xml = demo.attrs['model_file']
            model_xmls.append(model_xml.decode('utf-8') if isinstance(model_xml, bytes) else str(model_xml))

    print(f'exact two-goal bank: {len(names)} aligned initial/grasp/final triplets')
    return (torch.stack(starts).to(device), torch.stack(grasps).to(device), torch.stack(goals).to(device),
            names, initial_states, model_xmls)

def make_env(max_steps):
    controller = load_controller_config(default_controller='OSC_POSE')
    env = suite.make(
        'Stack_D0', robots='Panda', controller_configs=controller,
        has_renderer=False, has_offscreen_renderer=True,
        render_collision_mesh=False, render_visual_mesh=True,
        use_camera_obs=True, camera_names=[CAM_ABOVE, CAM_WRIST],
        camera_heights=IMG_SIZE, camera_widths=IMG_SIZE,
        control_freq=CONTROL_FREQ, horizon=max_steps, ignore_done=True,
    )
    set_geom_groups(env)
    return env

def reset_to_eval_state(env, model_xml, initial_state):
    env.reset()
    xml = env.edit_model_xml(model_xml)
    env.reset_from_xml_string(xml)
    env.sim.reset()
    env.sim.set_state_from_flattened(initial_state)
    env.sim.forward()
    set_geom_groups(env)
    return env._get_observations(force_update=True)

def render_frame(env):
    views = []
    for camera in (CAM_ABOVE, CAM_WRIST):
        view = env.sim.render(height=VIDEO_SIZE, width=VIDEO_SIZE, camera_name=camera)[::-1]
        views.append(np.asarray(view, dtype=np.uint8))
    return np.concatenate(views, axis=1)

def postprocess_action(action):
    action = np.clip(np.asarray(action, dtype=np.float64), -1.0, 1.0).copy()
    action[3:5] = 0.0  # Match the training collection contract.
    action[6] = 1.0 if action[6] >= 0 else -1.0
    return action

def is_cube_grasped(env):
    return bool(env._check_grasp(gripper=env.robots[0].gripper, object_geoms=env.cubeA))

def run_rollout(env, planner, encoder, start_embedding, grasp_embedding, stack_embedding,
                goal_name, initial_state, model_xml, grasp_confirm_steps, device, max_steps, video_path):
    video_path.parent.mkdir(parents=True, exist_ok=True)
    writer = imageio.get_writer(str(video_path), fps=CONTROL_FREQ, codec='libx264')
    try:
        obs = reset_to_eval_state(env, model_xml, initial_state)
        writer.append_data(render_frame(env))
        z0 = encode_obs(encoder, obs, device)
        print(f'exact eval: {goal_name} | restored-start embedding MSE: {(z0 - start_embedding).square().mean().item():.6f}')
        zi = torch.stack([z0] * WINDOW)
        past_actions = deque([torch.zeros(ACTION_DIM, dtype=z0.dtype, device=device) for _ in range(WINDOW - 1)], maxlen=WINDOW - 1)
        tier, grasp_streak = 'grasp', 0
        current_goal = grasp_embedding
        warm_start = None
        steps, success, done = 0, False, False
        print(f'initial-to-grasp latent MSE: {(z0 - grasp_embedding).square().mean().item():.4f}')
        while steps < max_steps:
            action_history = torch.stack(list(past_actions))
            action_sequence = np.asarray(planner.CEM(
                zi, action_history, current_goal, initial_mean=warm_start,
            ))
            warm_start = planner.shifted_plan(action_sequence)
            env_action = postprocess_action(action_sequence[0])
            obs, _, done, _ = env.step(env_action)
            set_geom_groups(env)
            writer.append_data(render_frame(env))
            steps += 1
            success = bool(env._check_success())
            past_actions.append(torch.as_tensor(env_action, dtype=z0.dtype, device=device))
            zt = encode_obs(encoder, obs, device)
            zi = torch.cat((zi[1:], zt[None]))
            if steps == 1 or steps % 25 == 0:
                goal_mse = (zt - current_goal).square().mean().item()
                print(f'step {steps:>3}/{max_steps} | {tier}-goal latent MSE: {goal_mse:.4f}')
            if done or success:
                break
            if tier == 'grasp':
                grasp_streak = grasp_streak + 1 if is_cube_grasped(env) else 0
                if grasp_streak >= grasp_confirm_steps:
                    tier = 'stack'
                    current_goal = stack_embedding
                    warm_start = None
                    print(f'red-cube grasp confirmed for {grasp_streak} steps; switched to final goal')
        return success
    finally:
        writer.close()


## Run simple two-goal inference

CEM receives only the saved grasp embedding during tier 1 and the final embedding during tier 2. The only active simulator privilege is the red-cube grasp predicate used for the one-way switch.

In [ ]:
NUM_ROLLOUTS = 5
MAX_STEPS = 400
EVAL_DEMO = None

(goal_starts, goal_grasps, goal_stacks, goal_names, initial_states, model_xmls) = load_tiered_eval_bank(
    encoder, EVAL_HDF5, DEVICE, EVAL_DEMO
)
# Generic controller support only; no statistics are read from evaluation actions.
action_mean = np.zeros(ACTION_DIM, dtype=np.float32)
action_std = np.array([0.5, 0.5, 0.5, 0.01, 0.01, 0.5, 1.0], dtype=np.float32)
action_low = np.array([-1.0, -1.0, -1.0, -0.01, -0.01, -1.0, -1.0], dtype=np.float32)
action_high = np.array([1.0, 1.0, 1.0, 0.01, 0.01, 1.0, 1.0], dtype=np.float32)
min_std = np.array([0.05, 0.05, 0.05, 0.005, 0.005, 0.05, 0.05], dtype=np.float32)
planner = Planner(
    DEVICE, encoder, predictor, HORIZON, NUM_SAMPLES, NUM_ELITES, CEM_ITERS, ACTION_DIM,
    discrete_gripper=True, action_mean=action_mean, action_std=action_std,
    action_low=action_low, action_high=action_high, min_std=min_std, momentum=0.1, seed=42,
)
print(f'two-goal planner H={planner.H}, N={planner.N}, K={planner.K}, T={planner.T}')
print('planning inputs: current history + grasp goal, then current history + final goal')

env = make_env(MAX_STEPS)
successes = 0
inference_start = time.perf_counter()
try:
    for rollout_index in range(NUM_ROLLOUTS):
        eval_index = rollout_index % len(goal_names)
        demo_name = goal_names[eval_index]
        video_path = ROLLOUT_DIR / f'{CHECKPOINT.stem}_{demo_name}_two_goal_rollout.mp4'
        success = run_rollout(
            env, planner, encoder, goal_starts[eval_index], goal_grasps[eval_index], goal_stacks[eval_index],
            demo_name, initial_states[eval_index], model_xmls[eval_index], GRASP_CONFIRM_STEPS,
            DEVICE, MAX_STEPS, video_path,
        )
        successes += int(success)
        print(f'rollout {rollout_index + 1}/{NUM_ROLLOUTS}: {"success" if success else "fail"}')
        print(f'video: {video_path}')
finally:
    env.close()
elapsed = time.perf_counter() - inference_start
print(f'success rate: {successes}/{NUM_ROLLOUTS} = {100 * successes / NUM_ROLLOUTS:.1f}%')
print(f'total inference time: {elapsed:.2f}s')

## Benchmark one full planning call

This measures one CEM call returning the complete `H x 7` action plan, without creating a robosuite environment.

In [ ]:
def synchronize():
    if DEVICE.startswith('cuda'):
        torch.cuda.synchronize()

benchmark_zi = torch.randn(WINDOW, EMBED_DIM, device=DEVICE)
benchmark_actions = torch.zeros(WINDOW - 1, ACTION_DIM, device=DEVICE)
benchmark_goal = torch.randn(EMBED_DIM, device=DEVICE)

print('warming up planner...')
_ = planner.CEM(benchmark_zi, benchmark_actions, benchmark_goal)
synchronize()
start = time.perf_counter()
planned_actions = planner.CEM(benchmark_zi, benchmark_actions, benchmark_goal)
synchronize()
planning_seconds = time.perf_counter() - start

print(f'one planning call: {planning_seconds:.3f}s')
print(f'action shape: {tuple(planned_actions.shape)}')
print(f'planner settings: H={planner.H}, N={planner.N}, K={planner.K}, T={planner.T}')